# Pelajaran 10 - Agen AI dalam Produksi

Dalam pelajaran ini Anda akan belajar **pola produksi** untuk agen AI menggunakan Microsoft Agent Framework (Python). Kami membahas:

- **Observabilitas** — menambahkan penjadwalan waktu dan pencatatan ke interaksi agen
- **Evaluasi** — menggunakan agen evaluator untuk menilai kualitas respon
- **Manajemen biaya** — strategi untuk optimasi token dan pemilihan model

Skenarionya adalah **agen perjalanan** yang membantu pengguna merencanakan perjalanan, dengan pemantauan dan evaluasi yang diterapkan di atasnya.


## Pengaturan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Pertimbangan Produksi

Memindahkan agen AI dari prototipe ke produksi memerlukan perhatian cermat pada tiga pilar:

1. **Observabilitas** — Anda perlu visibilitas ke dalam apa yang dilakukan agen, berapa lama waktu yang dibutuhkan, dan alat apa yang dipanggilnya. Tanpa pelacakan dan pencatatan, debugging masalah produksi hampir tidak mungkin dilakukan.

2. **Evaluasi** — Pemeriksaan kualitas otomatis memastikan respons agen tetap akurat, lengkap, dan membantu dari waktu ke waktu. Agen evaluator dapat menilai respons berdasarkan kriteria yang telah ditentukan.

3. **Manajemen Biaya** — Penggunaan token secara langsung memengaruhi biaya. Strategi seperti optimasi prompt, pemilihan model, dan caching membantu menjaga biaya tetap terkendali tanpa mengorbankan kualitas.


## Membangun Agen yang Dapat Diamati

Kami mendefinisikan alat perjalanan dan membungkus panggilan agen dengan penentuan waktu sehingga kami dapat memantau latensi. Di produksi, Anda akan mengintegrasikan dengan OpenTelemetry atau backend pelacakan serupa.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Pola Evaluasi

Pola produksi yang umum adalah menggunakan agen kedua sebagai **evaluator**. Evaluator memberikan skor pada respons agen utama berdasarkan kriteria yang telah ditentukan seperti kelengkapan, akurasi, dan kegunaan.

Ini memungkinkan:
- Pintu kualitas otomatis sebelum respons mencapai pengguna
- Deteksi regresi saat prompt atau model berubah
- Pemantauan berkelanjutan kinerja agen dari waktu ke waktu


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategi Manajemen Biaya

Mengendalikan biaya sangat penting untuk agen AI produksi. Berikut adalah strategi kunci:

| Strategi | Deskripsi |
|---|---|
| **Optimasi prompt** | Jaga instruksi sistem tetap ringkas. Hapus konteks yang berlebihan untuk mengurangi token input. |
| **Pemilihan model** | Gunakan model yang lebih kecil dan lebih murah (misalnya GPT-4o-mini) untuk tugas sederhana seperti klasifikasi atau ekstraksi, dan cadangkan model yang lebih besar untuk pemikiran yang kompleks. |
| **Caching** | Simpan hasil alat dan kueri yang sering digunakan untuk menghindari panggilan API yang berulang. |
| **Anggaran token** | Tetapkan batas `max_tokens` untuk mencegah respons yang tidak terduga panjang. |
| **Pengelompokan (Batching)** | Gabungkan beberapa kueri pengguna ke dalam satu panggilan API jika memungkinkan. |

Dalam praktiknya, pendekatan bertingkat bekerja dengan baik: arahkan permintaan yang sederhana ke model yang cepat dan murah dan hanya eskalasikan kueri yang kompleks ke model yang lebih mampu.


## Summary

Dalam pelajaran ini Anda belajar bagaimana untuk:

1. **Menambahkan observabilitas** pada interaksi agen dengan penjadwalan dan pencatatan, meletakkan dasar untuk pelacakan dan pemantauan.
2. **Mengevaluasi respons agen** secara otomatis menggunakan agen evaluator yang menilai kelengkapan, ketepatan, dan kegunaan.
3. **Mengelola biaya** melalui optimasi prompt, pemilihan model, caching, dan anggaran token.

Polapola produksi ini membantu memastikan agen AI Anda dapat diandalkan, terukur, dan hemat biaya dalam skala besar.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
